In [13]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
import numpy as np
import random
from qiskit.quantum_info import random_clifford, entropy, Clifford
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import Statevector, DensityMatrix, partial_trace
import matplotlib.pyplot as plt

In [14]:
gate = random_clifford(2, seed=None,)

def random_measurement(p, L, t):
    for i in range(L):
       if random.random() < p:
            cbit = L * t + i
            circuit.measure(i, cbit)
        
def random_brickwork_circuit(p, T, L):
    global circuit
    circuit = QuantumCircuit(L, T**2)

    for t in range(T):
        random_measurement(p, L, t)
        if t % 2 == 0:
            for i in range(int(L/2)):
                circuit.unitary(gate, [2*i, 2*i+1], label=" ")
        else:
            for i in range(int(L/2)):
                if 2*i == L-2:
                    circuit.unitary(gate, [L-1, 0], label=" ")
                else:
                    circuit.unitary(gate, [2*i+1, 2*i+2], label=" ")
    circuit.save_statevector()

circuit.draw("mpl")
random_brickwork_circuit(0.1, 10, 10)
    
def initial_density_matrix(L):
    initial_state = np.zeros(2 ** L)
    initial_state[0] = 1
    density_matrix = DensityMatrix(initial_state)
    return density_matrix

simulator = AerSimulator()
psi_not = initial_density_matrix(10)

average_entropy_array = []

def fig_3(p, T, L):
        rho_array = []
        for o in range(int(L/2)+1):
            rho_array.append(o+1)
        for i in range(20):
            random_brickwork_circuit(p, T, L)
            compiled_circuit = transpile(circuit, simulator)
            result = simulator.run(compiled_circuit).result()
            final_state = result.get_statevector()
            rho = DensityMatrix(final_state)
            rho_a = partial_trace(rho, rho_array)
            entanglement_entropy = entropy(rho_a)

            average_entropy_array.append(entanglement_entropy)
        return sum(average_entropy_array) / len(average_entropy_array)


fig_3_entropies_full = []

for q in range(1,6):
    fig_3_entropies = []
    for i in range(7):
        average_entropy = fig_3(i/7, 16*q, 8*q)
        fig_3_entropies.append(average_entropy)
    fig_3_entropies_full.append(fig_3_entropies)

y_8 = fig_3_entropies_full[:7]
y_16 = fig_3_entropies_full[7:13]
y_24 = fig_3_entropies_full[13:19]
y_40 = fig_3_entropies_full[31:37]

x = np.linspace(0, 1, 7)

plt.plot(x, y_8, '-o', label = 'L = 8')
plt.plot(x, y_16, '-o', label = 'L = 16')
plt.plot(x, y_24, '-o', label = 'L = 24')
plt.plot(x, y_40, '-o', label = 'L = 40')

plt.xlabel('p')
plt.ylabel('Half-Chain Stationary-State Entropy')
plt.show()

KeyboardInterrupt: 